# Устранение пропусков в данных

**Проблема.** Если в данных есть пропуски, то большинство алгоритмов машинного обучения не будет с ними работать. Даже корреляционная матрица не будет строиться корректно.

Существуют различные способы устранения пропусков в данных, которые связаны с удалением или заполнением пропусков.

## Загрузка и первичный анализ данных

Используем данные из соревнования [House Prices: Advanced Regression Techniques.](https://www.kaggle.com/c/house-prices-advanced-regression-techniques)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.impute import MissingIndicator
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from IPython.display import Image
%matplotlib inline 
sns.set(style="ticks")

In [ ]:
# Будем использовать только обучающую выборку
hdata_loaded = pd.read_csv('data/houseprices.csv', sep=",")

In [ ]:
hdata_loaded.shape

In [ ]:
hdata = hdata_loaded

## Удаление пропущенных значений

**Допущение:** пропуски распределены случайным образом.

**Когда рекомендуется использовать?**
- Если пропущенных данных слишком много и возникает опасность нарушить распределение исходных данных при заполнении пропусков. Рекомендуется удалять признак (колонку) целиком.
- Если датасет большой и пропущенных данных относительно немного, то рекомендуется удалять строки, содержащие пропуски в данных.
- Под "немного" в идеальном случае понимается 5% от выборки.

**Преимущества:**
- Простота реализации.
- При случайном распределении пропусков сохраняются параметры распределения исходных данных.

**Недостатки:**
- Может быть удален большой фрагмент данных при неудачном распределении пропусков в нескольких столбцах.
- Если пропуски распределены не случайно, то можно удалить значимые данные.

In [ ]:
list(zip(hdata.columns, [i for i in hdata.dtypes]))

In [ ]:
# Колонки с пропусками
hcols_with_na = [c for c in hdata.columns if hdata[c].isnull().sum() > 0]
hcols_with_na

In [ ]:
hdata.shape

In [ ]:
# Количество пропусков
[(c, hdata[c].isnull().sum()) for c in hcols_with_na]

In [ ]:
# Доля (процент) пропусков
[(c, hdata[c].isnull().mean()) for c in hcols_with_na]

In [ ]:
# Колонки для которых удаляются пропуски
hcols_with_na_temp = ['LotFrontage', 'GarageYrBlt', 'BsmtQual', 'MasVnrArea']

Удаление колонок, содержащих пустые значения
`res = data.dropna(axis=1, how='any')`

Удаление строк, содержащих пустые значения
`res = data.dropna(axis=0, how='any')`

[Документация](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.dropna.html)

**Удаление может производиться для группы строк или колонок.**

In [ ]:
# Удаление пропусков
hdata_drop = hdata[hcols_with_na_temp].dropna()
hdata_drop.shape

In [ ]:
def plot_hist_diff(old_ds, new_ds, cols):
    """
    Разница между распределениями до и после устранения пропусков
    """
    for c in cols:   
        fig = plt.figure()
        ax = fig.add_subplot(111)
        ax.title.set_text('Поле - ' + str(c))
        old_ds[c].hist(bins=50, ax=ax, density=True, color='green')
        new_ds[c].hist(bins=50, ax=ax, color='blue', density=True, alpha=0.5)
        plt.show()

In [ ]:
plot_hist_diff(hdata, hdata_drop, hcols_with_na_temp)

## Заполнение значений для одного признака

В этом случае данные которые находятся в соседних признаках  (колонках) не учитываются при заполнении.

**Заполнение (внедрение) значений или импьютация (imputation)** - это заполнение пропущенных значений их статистическими оценками.

**Для числовых признаков:**
- Заполнение [показателями центра распределения](https://ru.wikipedia.org/wiki/%D0%9F%D0%BE%D0%BA%D0%B0%D0%B7%D0%B0%D1%82%D0%B5%D0%BB%D0%B8_%D1%86%D0%B5%D0%BD%D1%82%D1%80%D0%B0_%D1%80%D0%B0%D1%81%D0%BF%D1%80%D0%B5%D0%B4%D0%B5%D0%BB%D0%B5%D0%BD%D0%B8%D1%8F).
- Заполнение константой. *Полезно в случае "неслучайного" распределения пропусков.*
- Заполнение "хвостом распределения".

**Для категориальных признаков:**
- Заполнение наиболее распространенным значением категории (аналогом моды).
- Введение отдельного значения категории для пропущенных значений.

**Для числовых и категориальных признаков:**
- Добавления флага пропусков.
- Заполнение случайным значением признака. *Метод обычно применяют на больших выборках. Преимуществом является то, что он сохраняет дисперсию исходной выборки.*



### Заполнение [показателями центра распределения](https://ru.wikipedia.org/wiki/%D0%9F%D0%BE%D0%BA%D0%B0%D0%B7%D0%B0%D1%82%D0%B5%D0%BB%D0%B8_%D1%86%D0%B5%D0%BD%D1%82%D1%80%D0%B0_%D1%80%D0%B0%D1%81%D0%BF%D1%80%D0%B5%D0%B4%D0%B5%D0%BB%D0%B5%D0%BD%D0%B8%D1%8F) и константой

**Для числовых признаков.**

В случае нормального распределения математическое ожидание, медиана и мода совпадают:

In [ ]:
Image('img/normal_dist.png', width='50%')

Для ассиметричных распределений эти показатели различаются:

In [ ]:
Image('img/center_skewed.png', width='30%')

**Когда рекомендуется использовать?**
- Если пропуски распределены случайным образом.
- В идеальном случае пропусков не более 5% от выборки.

**Какой показатель центра распределения лучше использовать?**
- Если распределение одномодальное, то лучше использовать моду, иначе математическое ожидание или медиану.
- Не существует однозначного предпочтения между математическим ожиданием или медианой. Но медиана более устойчива к выбросам в данных. 

Для внедрение) значений может быть использован класс [SimpleImputer.](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)

Для фильтрации пропущенных значений может быть использован класс [MissingIndicator.](https://scikit-learn.org/stable/modules/generated/sklearn.impute.MissingIndicator.html)

In [ ]:
# Пример работы MissingIndicator
temp_x1 = np.array([[np.nan, 1, 3], [4, 0, np.nan], [8, 1, 0]])
print('Исходный массив:')
print(temp_x1)
indicator = MissingIndicator()
temp_x1_transformed = indicator.fit_transform(temp_x1)
print('Маска пропущенных значений:')
print(temp_x1_transformed)

In [ ]:
def impute_column(dataset, column, strategy_param, fill_value_param=None):
    """
    Заполнение пропусков в одном признаке
    """
    temp_data = dataset[[column]].values
    size = temp_data.shape[0]
    
    indicator = MissingIndicator()
    mask_missing_values_only = indicator.fit_transform(temp_data)
    
    imputer = SimpleImputer(strategy=strategy_param, 
                            fill_value=fill_value_param)
    all_data = imputer.fit_transform(temp_data)
    
    missed_data = temp_data[mask_missing_values_only]
    filled_data = all_data[mask_missing_values_only]
    
    return all_data.reshape((size,)), filled_data, missed_data

In [ ]:
all_data, filled_data, missed_data = impute_column(hdata, 'MasVnrArea', 'mean')
all_data

In [ ]:
filled_data

In [ ]:
missed_data

In [ ]:
def research_impute_numeric_column(dataset, num_column, const_value=None):
    strategy_params = ['mean', 'median', 'most_frequent', 'constant']
    strategy_params_names = ['Среднее', 'Медиана', 'Мода']
    strategy_params_names.append('Константа = ' + str(const_value))
    
    original_temp_data = dataset[[num_column]].values
    size = original_temp_data.shape[0]
    original_data = original_temp_data.reshape((size,))
    
    new_df = pd.DataFrame({'Исходные данные':original_data})
       
    for i in range(len(strategy_params)):
        strategy = strategy_params[i]
        col_name = strategy_params_names[i]
        if (strategy!='constant') or (strategy == 'constant' and const_value!=None):
            if strategy == 'constant':
                temp_data, _, _ = impute_column(dataset, num_column, strategy, fill_value_param=const_value)
            else:
                temp_data, _, _ = impute_column(dataset, num_column, strategy)
            new_df[col_name] = temp_data
        
    sns.kdeplot(data=new_df)

In [ ]:
research_impute_numeric_column(hdata, 'GarageYrBlt', 1950)

In [ ]:
research_impute_numeric_column(hdata, 'LotFrontage')

In [ ]:
research_impute_numeric_column(hdata, 'MasVnrArea')

### Заполнение "хвостом распределения"

**Для числовых признаков.**

**Допущение:** пропуски распределены НЕ случайным образом. Мы хотим выделить пропущенные значения из остальных значений.

**Преимущества:**
- Простота реализации.
- Выделяет пропущенные значения из остальных значений.

**Недостатки:**
- Нарушение параметров исходного распределения.
- Поскольку значения на краю распределения фактически являются аномалиями (выбросами), то данный подход может пересекаться с алгоритмами поиска аномалий.

**Как вычислить "хвост распределения"?**

Если распределение данных признака $f$ напоминает нормальное:

$$ extreme\_value = mean(f) + 3 \cdot std(f) $$

Для ассиметричного распределения:

$$ IQR = Q3-Q1 $$

$IQR -$ interquartile range.

$$ extreme\_value = Q3 + K \cdot IQR $$

Значение $K$ обычно выбирается равным $1,5$. Но для экстремальных выбросов выбирают $K=3$.



In [ ]:
# Похоже на нормальное
LotFrontage_ev = hdata['LotFrontage'].mean() + 3*hdata['LotFrontage'].std()
LotFrontage_ev

In [ ]:
research_impute_numeric_column(hdata, 'LotFrontage', LotFrontage_ev)

In [ ]:
# Ассиметричное
IQR = hdata['MasVnrArea'].quantile(0.75) - hdata['MasVnrArea'].quantile(0.25)
MasVnrArea_ev1 = hdata['MasVnrArea'].quantile(0.75) + 3*IQR
print('IQR={}, extreme_value={}'.format(IQR, MasVnrArea_ev1))

In [ ]:
research_impute_numeric_column(hdata, 'MasVnrArea', MasVnrArea_ev1)

In [ ]:
MasVnrArea_ev2 = hdata['MasVnrArea'].quantile(0.75) + 1.5*IQR
print('IQR={}, extreme_value={}'.format(IQR, MasVnrArea_ev2))

In [ ]:
research_impute_numeric_column(hdata, 'MasVnrArea', MasVnrArea_ev2)

In [ ]:
IQR_lf = hdata['LotFrontage'].quantile(0.75) - hdata['LotFrontage'].quantile(0.25)
LotFrontage_ev1 = hdata['LotFrontage'].quantile(0.75) + 1.5*IQR_lf
LotFrontage_ev2 = hdata['LotFrontage'].quantile(0.75) + 3*IQR_lf

In [ ]:
research_impute_numeric_column(hdata, 'LotFrontage', LotFrontage_ev1)

In [ ]:
research_impute_numeric_column(hdata, 'LotFrontage', LotFrontage_ev2)

### Заполнение наиболее распространенным значением категории

**Для категориальных признаков.**

**Допущение:** пропуски распределены случайным образом. Заполнение пропусков наиболее распространенным значением категории в наименьшей степени повлияет на исходное распределение.

In [ ]:
hdata_cat_cols = ['GarageType', 'PoolQC', 'Fence']
hdata_cat_new = hdata[hdata_cat_cols].copy() 

In [ ]:
GarageType_cat_new_temp, _, _ = impute_column(hdata_cat_new, 'GarageType', 'most_frequent')
PoolQC_cat_new_temp, _, _ = impute_column(hdata_cat_new, 'PoolQC', 'most_frequent')
Fence_cat_new_temp, _, _ = impute_column(hdata_cat_new, 'Fence', 'most_frequent')

In [ ]:
hdata_cat_new['GarageType'] = GarageType_cat_new_temp
hdata_cat_new['PoolQC'] = PoolQC_cat_new_temp
hdata_cat_new['Fence'] = Fence_cat_new_temp

In [ ]:
plot_hist_diff(hdata, hdata_cat_new, hdata_cat_cols)

### Введение отдельного значения категории для пропущенных значений

**Для категориальных признаков.**

**Основное преимущество подхода** состоит в том, что не дается никаких предположений о распределении  пропущенных значений.

In [ ]:
hdata_cat_na = hdata[hdata_cat_cols].copy()

In [ ]:
GarageType_cat_na_temp, _, _ = impute_column(hdata_cat_na, 'GarageType', 'constant', fill_value_param='NA')
PoolQC_cat_na_temp, _, _ = impute_column(hdata_cat_na, 'PoolQC', 'constant', fill_value_param='NA')
Fence_cat_na_temp, _, _ = impute_column(hdata_cat_na, 'Fence', 'constant', fill_value_param='NA')

In [ ]:
hdata_cat_na['GarageType'] = GarageType_cat_na_temp
hdata_cat_na['PoolQC'] = PoolQC_cat_na_temp
hdata_cat_na['Fence'] = Fence_cat_na_temp

In [ ]:
plot_hist_diff(hdata, hdata_cat_na, hdata_cat_cols)

### Добавления флага пропусков

**Для любых признаков.**

Для каждой колонки данных вводится дополнительная бинарная колонка, в которой пустым значениям признака соответствует 1.

**Преимущества:**
- Модель получает дополнительную информацию о том, насколько мы уверены в наших данных. Особенно хорошо этот подход работает для деревьев решений и производных моделей.

**Недостатки:**
- Расширяется признаковое пространство.
- Заполнять пропуски для исходных колонок все равно необходимо.
- Флаги для разных колонок могут сильно коррелировать между собой.

In [ ]:
hdata_mis = hdata[['PoolQC']].copy()
hdata_mis.head()

In [ ]:
indicator = MissingIndicator()
PoolQC_missing = indicator.fit_transform(hdata_mis[['PoolQC']])
PoolQC_missing

In [ ]:
PoolQC_missing_int = [1 if i==True else 0 for i in PoolQC_missing]
PoolQC_missing_int[:10]

In [ ]:
hdata_mis['PoolQC_missing'] = PoolQC_missing_int
hdata_mis.head()

## Заполнение значений для нескольких признаков

В этом случае данные которые находятся в соседних признаках (колонках) учитываются при заполнении.

Идея состоит в том, что признаки могут зависеть между собой и такие зависимости необходимо использовать при заполнении пропусков.

В этом случае мы решаем отдельную задачу машинного обучения, рассматривая пропущенный признак как целевой (y), а остальные признаки как исходные (X).

Для решения задачи можно использовать различные методы машинного обучения. На практике чаще всего используется **метод ближайших соседей**.

Также проблема состоит в том, что практически все признаки могут содержать пропуски и для их заполнения другие признаки необходимо предварительно импьютировать известными методами. (возникает подобие циклических ссылок).

Для решения этой задачи используется метод 
**MICE (multivariate Imputation of Chained Equations)**. Существует расширение этого метода **MissForest** в котором используется случайный лес.

### Импьютация с использованием метода  ближайших соседей

Используется класс [KNNImputer.](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html)

In [ ]:
knnimpute_cols = [
    'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
    'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea',
    'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
    'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
    'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea',
    'WoodDeckSF',  'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
    'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold',
    'SalePrice'
]

In [ ]:
knnimpute_hdata = hdata[knnimpute_cols].copy()
knnimpute_hdata.head()

In [ ]:
# Признаки с пропусками
knnimpute_hdata.isnull().sum()

In [ ]:
knnimputer = KNNImputer(
    n_neighbors=5, 
    weights='distance', 
    metric='nan_euclidean', 
    add_indicator=False, 
)
knnimpute_hdata_imputed_temp = knnimputer.fit_transform(knnimpute_hdata)
knnimpute_hdata_imputed = pd.DataFrame(knnimpute_hdata_imputed_temp, columns=knnimpute_hdata.columns)
knnimpute_hdata_imputed.head()

In [ ]:
# Пропуски заполнены
knnimpute_hdata_imputed.isnull().sum()

In [ ]:
LotFrontage_df = pd.DataFrame({'original': knnimpute_hdata['LotFrontage'].values})
LotFrontage_df['KNN_5'] = knnimpute_hdata_imputed['LotFrontage']
sns.kdeplot(data=LotFrontage_df)

In [ ]:
GarageYrBlt_df = pd.DataFrame({'original': knnimpute_hdata['GarageYrBlt'].values})
GarageYrBlt_df['KNN_5'] = knnimpute_hdata_imputed['GarageYrBlt']
sns.kdeplot(data=GarageYrBlt_df)

#### Подбор гиперпараметров

Так как для импьютации используется модель KNN, то возникает необходимость подбора гиперпараметров. В этом случае необходимо создать полный пайплайн машинного обучения и оптимизировать параметры всего пайплайна.

In [ ]:
pipe = Pipeline(steps=[
    ('imputer', KNNImputer(
        n_neighbors=5,
        weights='distance',
        add_indicator=False)),
    ('scaler', StandardScaler()),
    ('regressor', Lasso(max_iter=2000)),
])

In [ ]:
param_grid = {
    'imputer__n_neighbors': [3,5,10],
    'imputer__weights': ['uniform', 'distance'],
    'imputer__add_indicator': [True, False],
    'regressor__alpha': [10, 100, 200],
}

In [ ]:
grid_search = GridSearchCV(pipe, param_grid, cv=5, n_jobs=-1, scoring='r2')

In [ ]:
grid_search.fit(knnimpute_hdata, hdata['SalePrice'])

In [ ]:
grid_search.best_params_

In [ ]:
knnimputer2 = KNNImputer(
    n_neighbors=grid_search.best_params_['imputer__n_neighbors'], 
    weights=grid_search.best_params_['imputer__weights'], 
    metric='nan_euclidean', 
    add_indicator=False, 
)
knnimpute_hdata_imputed_temp2 = knnimputer2.fit_transform(knnimpute_hdata)
knnimpute_hdata_imputed2 = pd.DataFrame(knnimpute_hdata_imputed_temp2, columns=knnimpute_hdata.columns)
knnimpute_hdata_imputed2.head()

In [ ]:
LotFrontage_df['KNN_after_gridsearch'] = knnimpute_hdata_imputed2['LotFrontage']
sns.kdeplot(data=LotFrontage_df)

In [ ]:
GarageYrBlt_df['KNN_after_gridsearch'] = knnimpute_hdata_imputed2['GarageYrBlt']
sns.kdeplot(data=GarageYrBlt_df)

### Метод  MICE (multivariate Imputation of Chained Equations)

Описание метода и его реализация в пакете [statsmodels.](https://www.statsmodels.org/stable/imputation.html)

**Основные шаги метода:**
1. Используем произвольную импьютацию для всех колонок (например, для числовых колонок - средним значением).
1. Для первой колонки подставляем обратно пустые значения.
1. Пропущенные значения в первой колонке предсказываются с помощью модели машинного обучения  на основе остальных значений.
1. Предсказанные значения подставляются в первую колонку.
1. Действия выше выполняются в цикле для всех колонок.
1. Действия выше выполняются заданное количество раз. Предполагается что на каждой итерации мы постепенно уточняем значения, соответствующие пропускам.


Рассмотрим реализацию [MissForest](https://towardsdatascience.com/missforest-the-best-missing-data-imputation-algorithm-4d01182aed3) с использованием [IterativeImputer.](https://scikit-learn.org/stable/modules/generated/sklearn.impute.IterativeImputer.html)

In [ ]:
imputer_missForest = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=10, random_state=0),
    max_iter=10,
    random_state=0)

In [ ]:
%%time
missForest_hdata_imputed_temp = imputer_missForest.fit_transform(knnimpute_hdata)

In [ ]:
missForest_hdata_imputed = pd.DataFrame(missForest_hdata_imputed_temp, columns=knnimpute_hdata.columns)
missForest_hdata_imputed.head()

In [ ]:
LotFrontage_df['MissForest'] = missForest_hdata_imputed['LotFrontage']
sns.kdeplot(data=LotFrontage_df)

In [ ]:
GarageYrBlt_df['MissForest'] = missForest_hdata_imputed['GarageYrBlt']
sns.kdeplot(data=GarageYrBlt_df)

Методы KNNImputer и MissForest также реализованы в библиотеке [missingpy.](https://github.com/epsilon-machine/missingpy)